# CatMap: photos to CSV

Reads every photo in `photos_raw/` , writes one row per photo to `data/cat_photos.csv`, with the photo's coordinates and the date it was taken. Open the CSV in Excel, Numbers or Google Sheets to look through it and fill in names and descriptions.

```
geog464-starter/
  photos_raw/                   your exported iPhone photos
  data/cat_photos.csv           output, written by this notebook
  notebooks/photos_to_csv.ipynb this notebook
  site/photos/                  web-ready JPEGs for the map (step 6)
```

Run the cells top to bottom. Re-running after you add photos is safe: anything you typed in `name`, `category` or `description` is kept.

## 0) One-time setup
iPhone photos are usually `.HEIC`, which needs the `pillow-heif` plugin. Run this once, then restart the kernel.

In [1]:
%pip install pillow pillow-heif pandas

Note: you may need to restart the kernel to use updated packages.


## 1) Settings
Paths are relative to the `notebooks/` folder, like in `mvp_pipeline.ipynb`.

In [7]:
from pathlib import Path
from datetime import datetime
import pandas as pd
from PIL import Image

try:
    from pillow_heif import register_heif_opener
    register_heif_opener()
except ImportError:
    print("pillow-heif is not installed, so .HEIC photos can't be read. See step 0.")

PHOTO_DIR = Path("../photos_raw")
CSV_PATH = Path("../data/cat_photos.csv")
PHOTO_EXTENSIONS = {".jpg", ".jpeg", ".heic", ".heif", ".png"}

## 2) Read the location and date from one photo
EXIF stores coordinates as degrees, minutes and seconds plus a hemisphere letter (N/S, E/W). `dms_to_decimal` turns them into the decimal degrees maps use. Montréal is west of Greenwich, so longitudes come out negative (around -73.6).

In [8]:
GPS_IFD = 0x8825   # EXIF block with the GPS tags
EXIF_IFD = 0x8769  # EXIF block with the date the photo was taken


def dms_to_decimal(dms, ref):
    degrees, minutes, seconds = (float(x) for x in dms)
    value = degrees + minutes / 60 + seconds / 3600
    return -value if str(ref).strip().upper() in ("S", "W") else value


def read_metadata(path):
    # Returns (lat, lon, date). Each is None if the photo doesn't have it.
    with Image.open(path) as img:
        exif = img.getexif()

    lat = lon = None
    gps = exif.get_ifd(GPS_IFD)
    if 2 in gps and 4 in gps:  # 2 = latitude, 4 = longitude
        lat = round(dms_to_decimal(gps[2], gps.get(1, "N")), 6)
        lon = round(dms_to_decimal(gps[4], gps.get(3, "E")), 6)
        if lat == 0 and lon == 0:  # some apps write 0,0 instead of leaving it blank
            lat = lon = None

    raw_date = exif.get_ifd(EXIF_IFD).get(36867) or exif.get(306)  # date taken, else date modified
    date = None
    if raw_date:
        try:
            date = datetime.strptime(str(raw_date).strip(), "%Y:%m:%d %H:%M:%S").strftime("%Y-%m-%d %H:%M")
        except ValueError:
            date = str(raw_date).strip()
    return lat, lon, date

## 3) Go through every photo in the folder
Subfolders are included. Photos without a location still get a row, with `lat` and `lon` left blank, so you can see which ones are missing and fill them in by hand if you want.

In [9]:
photo_files = sorted(
    p for p in PHOTO_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in PHOTO_EXTENSIONS and not p.name.startswith(".")
)
print(f"Found {len(photo_files)} photos in {PHOTO_DIR}")

rows = []
for path in photo_files:
    try:
        lat, lon, date = read_metadata(path)
    except Exception as e:
        print(f"  Could not read {path.name}: {e}")
        continue
    rows.append({
        "photo": path.name,
        "date": date,
        "lat": lat,
        "lon": lon,
        "name": "",
        "category": "Cat",
        "description": "",
    })

df = pd.DataFrame(rows, columns=["photo", "date", "lat", "lon", "name", "category", "description"])
has_gps = df["lat"].notna()
print(f"With location: {has_gps.sum()}   Without location: {(~has_gps).sum()}")

Found 75 photos in ../photos_raw
With location: 73   Without location: 2


## 4) Save the CSV
If `cat_photos.csv` already exists, your edits are carried over: `name`, `category` and `description`, plus any `lat`/`lon` you typed in for photos that had no location.

In [10]:
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

if CSV_PATH.exists():
    old = pd.read_csv(CSV_PATH, keep_default_na=False, dtype=str).set_index("photo")
    for col in ["name", "category", "description", "lat", "lon"]:
        if col not in old.columns:
            continue
        typed = df["photo"].map(old[col])
        keep = typed.notna() & (typed != "")
        if col in ("lat", "lon"):
            keep &= df[col].isna()  # only fill blanks, never overwrite GPS from the photo
            typed = pd.to_numeric(typed, errors="coerce")
        df[col] = df[col].where(~keep, typed)

df = df.sort_values("date", na_position="last").reset_index(drop=True)
df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")  # utf-8-sig so Excel shows accents correctly
print(f"Saved {len(df)} photos to {CSV_PATH}")
df

Saved 75 photos to ../data/cat_photos.csv


,photo,date,lat,lon,name,category,description
0,IMG_8621.HEIC,2024-05-15 14:24,45.520292,-73.580367,,Cat,
1,IMG_8914.HEIC,2024-05-25 13:45,45.522014,-73.581261,,Cat,
2,IMG_1565.HEIC,2024-06-09 10:50,45.521950,-73.581153,,Cat,
3,IMG_0173.HEIC,2024-06-30 19:37,45.520656,-73.581300,,Cat,
4,FullSizeRender.HEIC,2024-06-30 20:03,45.521992,-73.581506,,Cat,
...,...,...,...,...,...,...,...
70,IMG_5501.HEIC,2026-09-26 16:55,45.523422,-73.588272,,Cat,
71,IMG_5504.HEIC,2026-09-26 19:24,45.529969,-73.586731,,Cat,
72,IMG_5703.jpeg,2026-10-07 17:25,45.516858,-73.576125,,Cat,
73,IMG_5938.HEIC,2026-10-19 12:12,45.521767,-73.601503,,Cat,


## 5) Check for photos outside Montréal
Anything listed here is probably from a trip. Delete those rows or remove the photos from the folder.

In [11]:
outside = df[df["lat"].notna() & ~(df["lat"].between(45.35, 45.75) & df["lon"].between(-74.0, -73.4))]
print(f"{len(outside)} photo(s) outside Montréal")
outside

0 photo(s) outside Montréal


,photo,date,lat,lon,name,category,description


## 6) Make web copies of the photos for the map
Browsers can't show iPhone `.HEIC` files, so this saves a smaller JPEG of every photo in `site/photos/`, which is what the map popups display. The copies are rotated the right way up and have their metadata removed, so the published images don't carry GPS data. Photos already copied are skipped, so re-running is quick.

In [12]:
from PIL import ImageOps

WEB_DIR = Path("../site/photos")
WEB_MAX_SIZE = 1200  # longest side in pixels
WEB_DIR.mkdir(parents=True, exist_ok=True)

stems = pd.Series([p.stem for p in photo_files])
dupes = sorted(stems[stems.duplicated()].unique())
if dupes:
    print(f"Warning: these file names appear more than once and will overwrite each other: {dupes}")

made = 0
for path in photo_files:
    dest = WEB_DIR / (path.stem + ".jpg")
    if dest.exists() and dest.stat().st_mtime >= path.stat().st_mtime:
        continue
    try:
        with Image.open(path) as img:
            img = ImageOps.exif_transpose(img).convert("RGB")  # apply the iPhone's rotation
            img.thumbnail((WEB_MAX_SIZE, WEB_MAX_SIZE))
            img.save(dest, "JPEG", quality=85, optimize=True)  # saved without metadata
        made += 1
    except Exception as e:
        print(f"  Could not copy {path.name}: {e}")
print(f"Made {made} new web copies in {WEB_DIR}")

Made 0 new web copies in ../site/photos
